# Qwen3-4B-Instruct-2507 Baseline Evaluation

**Purpose**: Establish baseline performance metrics for the generation model on Vietnamese medical QA

**Model**: Qwen/Qwen3-4B-Instruct-2507

**Dataset**: combined_medical_qa_dataset

**Metrics**:
- BLEU score
- ROUGE-L (F1, Precision, Recall)
- BERTScore (F1, Precision, Recall)

**Output**: Baseline metrics logged to W&B and saved to `ml/results/generation_baseline_metrics.json`

## 1. Setup and Imports

In [ ]:
import os
import json
from pathlib import Path
from typing import List, Dict

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_from_disk
import wandb
from tqdm import tqdm
import numpy as np

# Evaluation metrics
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## 2. Configuration

In [ ]:
# Paths
DATA_DIR = Path("../../data/combined_medical_qa")
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Model configuration
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

# Evaluation configuration (from Qwen3 guidelines)
MAX_EVAL_SAMPLES = 100  # Set to None to evaluate all test samples
MAX_NEW_TOKENS = 16384  # Qwen team recommends 16K for most queries
TEMPERATURE = 0.7  # Recommended by Qwen team
TOP_P = 0.8  # Recommended by Qwen team
TOP_K = 20  # Recommended by Qwen team

# W&B configuration
WANDB_PROJECT = "vietnamese-medical-rag"
WANDB_RUN_NAME = "qwen3-4b-baseline"

print(f"Model: {MODEL_NAME}")
print(f"Dataset: {DATA_DIR}")
print(f"Max eval samples: {MAX_EVAL_SAMPLES or 'All'}")
print(f"Using Qwen3 recommended parameters: temp={TEMPERATURE}, top_p={TOP_P}, top_k={TOP_K}")

## 3. Load Dataset

In [ ]:
# Load dataset from disk
print(f"Loading dataset from {DATA_DIR}...")
dataset = load_from_disk(str(DATA_DIR))

# Check if test split exists, if not, create it
if "test" not in dataset:
    print("⚠️  No test split found. Creating train/validation/test splits...")
    # Split: 80% train, 10% validation, 10% test
    train_test = dataset["train"].train_test_split(test_size=0.2, seed=42)
    val_test = train_test["test"].train_test_split(test_size=0.5, seed=42)
    
    dataset = {
        "train": train_test["train"],
        "validation": val_test["train"],
        "test": val_test["test"]
    }
    print(f"✓ Created splits - Train: {len(dataset['train'])}, Val: {len(dataset['validation'])}, Test: {len(dataset['test'])}")
else:
    print("✓ Found existing test split")

# Get test split
test_dataset = dataset["test"]

# Limit evaluation samples if specified
if MAX_EVAL_SAMPLES:
    test_dataset = test_dataset.select(range(min(MAX_EVAL_SAMPLES, len(test_dataset))))

print(f"\nTest samples: {len(test_dataset)}")
print(f"\nSample data:")
print(test_dataset[0])

## 4. Load Model

In [ ]:
# Load tokenizer
print(f"Loading tokenizer from {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load model
print(f"Loading model from {MODEL_NAME}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print(f"\nModel loaded successfully!")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

## 5. Generate Predictions

In [ ]:
def generate_response(question: str, context: str = None) -> str:
    """Generate response for a given question using Qwen3 chat template."""
    # Format as chat messages (Qwen3 best practice)
    if context:
        user_message = f"""Dựa vào ngữ cảnh sau, hãy trả lời câu hỏi.

Ngữ cảnh: {context}

Câu hỏi: {question}"""
    else:
        user_message = f"Hãy trả lời câu hỏi sau: {question}"
    
    messages = [
        {"role": "system", "content": "Bạn là trợ lý y tế AI chuyên nghiệp."},
        {"role": "user", "content": user_message}
    ]
    
    # Apply chat template (CRITICAL: Qwen3 requires this)
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # Tokenize
    inputs = tokenizer([text], return_tensors="pt").to(device)
    
    # Generate with Qwen3 recommended parameters
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id
        )
    
    # Decode only the generated part
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()


# Generate predictions
print("Generating predictions with Qwen3-4B-Instruct-2507...")
print("Using Qwen3 chat template and recommended sampling parameters")
predictions = []
references = []

for sample in tqdm(test_dataset):
    question = sample["question"]
    context = sample.get("context", None)
    reference = sample["answer"]
    
    prediction = generate_response(question, context)
    
    predictions.append(prediction)
    references.append(reference)

print(f"\nGenerated {len(predictions)} predictions")
print(f"\nSample prediction:")
print(f"Question: {test_dataset[0]['question']}")
print(f"Reference: {references[0]}")
print(f"Prediction: {predictions[0]}")

## 6. Calculate Metrics

In [ ]:
def calculate_bleu(predictions: List[str], references: List[str]) -> float:
    """Calculate average BLEU score."""
    smoothing = SmoothingFunction().method1
    scores = []
    
    for pred, ref in zip(predictions, references):
        score = sentence_bleu(
            [ref.split()],
            pred.split(),
            smoothing_function=smoothing
        )
        scores.append(score)
    
    return np.mean(scores)


def calculate_rouge(predictions: List[str], references: List[str]) -> Dict:
    """Calculate ROUGE-L scores."""
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
    
    precision_scores = []
    recall_scores = []
    f1_scores = []
    
    for pred, ref in zip(predictions, references):
        score = scorer.score(ref, pred)
        precision_scores.append(score['rougeL'].precision)
        recall_scores.append(score['rougeL'].recall)
        f1_scores.append(score['rougeL'].fmeasure)
    
    return {
        "precision": np.mean(precision_scores),
        "recall": np.mean(recall_scores),
        "f1": np.mean(f1_scores)
    }


def calculate_bertscore(predictions: List[str], references: List[str]) -> Dict:
    """Calculate BERTScore."""
    P, R, F1 = bert_score(predictions, references, lang="vi", verbose=False)
    
    return {
        "precision": P.mean().item(),
        "recall": R.mean().item(),
        "f1": F1.mean().item()
    }


# Calculate all metrics
print("Calculating BLEU...")
bleu_score = calculate_bleu(predictions, references)

print("Calculating ROUGE-L...")
rouge_scores = calculate_rouge(predictions, references)

print("Calculating BERTScore...")
bert_scores = calculate_bertscore(predictions, references)

# Compile results
metrics = {
    "model": MODEL_NAME,
    "dataset": "combined_medical_qa_dataset",
    "num_samples": len(predictions),
    "bleu": bleu_score,
    "rouge_l": rouge_scores,
    "bert_score": bert_scores
}

print("\n" + "="*70)
print("BASELINE METRICS")
print("="*70)
print(f"BLEU: {bleu_score:.4f}")
print(f"ROUGE-L F1: {rouge_scores['f1']:.4f}")
print(f"ROUGE-L Precision: {rouge_scores['precision']:.4f}")
print(f"ROUGE-L Recall: {rouge_scores['recall']:.4f}")
print(f"BERTScore F1: {bert_scores['f1']:.4f}")
print(f"BERTScore Precision: {bert_scores['precision']:.4f}")
print(f"BERTScore Recall: {bert_scores['recall']:.4f}")
print("="*70)

## 7. Log to W&B

In [ ]:
# Initialize W&B
wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    config={
        "model": MODEL_NAME,
        "dataset": "combined_medical_qa_dataset",
        "max_new_tokens": MAX_NEW_TOKENS,
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "num_samples": len(predictions)
    }
)

# Log metrics
wandb.log({
    "bleu": bleu_score,
    "rouge_l_f1": rouge_scores['f1'],
    "rouge_l_precision": rouge_scores['precision'],
    "rouge_l_recall": rouge_scores['recall'],
    "bert_score_f1": bert_scores['f1'],
    "bert_score_precision": bert_scores['precision'],
    "bert_score_recall": bert_scores['recall']
})

# Log sample predictions
table = wandb.Table(columns=["Question", "Reference", "Prediction"])
for i in range(min(10, len(predictions))):
    table.add_data(
        test_dataset[i]["question"],
        references[i],
        predictions[i]
    )
wandb.log({"sample_predictions": table})

wandb.finish()
print("\nMetrics logged to W&B successfully!")

## 8. Save Results

In [ ]:
# Save metrics to file
output_path = RESULTS_DIR / "generation_baseline_metrics.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print(f"\nBaseline metrics saved to: {output_path}")

# Save predictions for analysis
predictions_path = RESULTS_DIR / "generation_baseline_predictions.json"
with open(predictions_path, "w", encoding="utf-8") as f:
    json.dump({
        "predictions": predictions,
        "references": references,
        "questions": [sample["question"] for sample in test_dataset]
    }, f, indent=2, ensure_ascii=False)

print(f"Predictions saved to: {predictions_path}")
print("\n✓ Baseline evaluation complete!")